In [ ]:
from IPython import get_ipython
from matplotlib import pyplot as plt
import numpy as np
from scipy.sparse import csc_matrix
from scipy import io
from caiman.source_extraction.cnmf.cnmf import load_CNMF
from caiman.base.rois import register_multisession
from caiman.utils import visualization
from caiman.utils.utils import download_demo
from pathlib import Path
import os
import re
import pickle
from datetime import datetime

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass

In [ ]:
def load_cnmf_data(file_paths):
    """
    Load CNMF data and extract spatial and template information.
    
    Parameters:
    - file_paths (list of str): List of file paths to the CNMF result files.
    
    Returns:
    - spatial (list): List of spatial components matrices (A) for each session.
    - templates (list): List of templates (Cn) for each session.
    - dims (tuple): Dimensions of the spatial layout from the last loaded file.
    """
    cnmf_list=[]
    spatial = []
    templates = []
    dims = None

    for file_path in file_paths:
        print(file_path)
        cnmf_model = load_CNMF(str(file_path))
        cnmf_list.append(cnmf_model)
        idx_components = cnmf_model.estimates.idx_components  # Idx only for good components
        spatial.append(cnmf_model.estimates.A[:, idx_components])  # Spatial components (A) ## is it A that we should use? Took it from the demo ipynb
        templates.append(cnmf_model.estimates.Cn)  # Template (Cn)  ## is it Cv that we should use? only variable that has the correct dims
        dims = cnmf_model.estimates.dims  # Spatial dimensions

    return cnmf_list, spatial, templates, dims, idx_components


### Day and Task Effects on Registration

In [ ]:
# Test 
# Define file paths to.hdf5 files
file_paths=[]
mouse_id = 1637
hard_disk = "T7Shield"
round = 3
date_format = "%Y-%m-%dT%H_%M_%S" 


# TMaze directories
task = "TMaze" # "TMaze" or "Linear"
directory_path_1 = f"/media/toor/{hard_disk}/Mouse{mouse_id}/Round{round}/{task}"
print(directory_path_1)
session_dates = [name for name in os.listdir(directory_path_1) if os.path.isdir(os.path.join(directory_path_1, name))]
[session_dates.remove(x) for x in ['Cellreg_processed', 'Caiman_Registration', 'FirstRound', 'Round1.5', 'caiman_awaiting'] if x in session_dates]
sorted_session_dates = sorted(session_dates, key=lambda date: datetime.strptime(date, date_format))
TMaze_file_paths = [f"/media/toor/{hard_disk}/Mouse{mouse_id}/Round{round}/{task}/" + session_date + "/caiman_final/caiman_results.hdf5" for session_date in sorted_session_dates]

# Linear directories 
task = "Linear" # "TMaze" or "Linear"
directory_path_1 = f"/media/toor/{hard_disk}/Mouse{mouse_id}/Round{round}/{task}"
print(directory_path_1)
session_dates = [name for name in os.listdir(directory_path_1) if os.path.isdir(os.path.join(directory_path_1, name))]
[session_dates.remove(x) for x in ['Cellreg_processed', 'Caiman_Registration', 'FirstRound', 'Round1.5', 'caiman_awaiting'] if x in session_dates]
sorted_session_dates = sorted(session_dates, key=lambda date: datetime.strptime(date, date_format))
Linear_file_paths = [f"/media/toor/{hard_disk}/Mouse{mouse_id}/Round{round}/{task}/" + session_date + "/caiman_final/caiman_results.hdf5" for session_date in sorted_session_dates]


same_taskTMaze_differentDay = TMaze_file_paths[4:6]
same_taskLinear_differentDay = Linear_file_paths[5:7]
cross_task_sameDay = Linear_file_paths[5:6] + TMaze_file_paths[4:5]
cross_task_differentDay = Linear_file_paths[5:6] + TMaze_file_paths[5:6]

print("Same task T-Maze file paths:", same_taskTMaze_differentDay)
print("Same task Linear file paths:", same_taskLinear_differentDay)   
print("Cross task same day file paths:", cross_task_sameDay)
print("Cross task different day file paths:", cross_task_differentDay)

In [ ]:
# load data
cnmf_list, spatial, templates, dims, idx_components = load_cnmf_data(same_taskTMaze_differentDay)
# multisession registration
spatial_union, assignments, matchings = register_multisession(A=spatial, dims=dims, templates=templates)
# Save results
save_dir = f"/media/toor/T7Shield/Mouse1637/Registration_CrossTask_Tests/"

# Ensure the directory exists
os.makedirs(save_dir, exist_ok=True)

np.save(f"{save_dir}spatial_union_same_taskTMaze_differentDay.npy", spatial_union)
np.save(f"{save_dir}assignments_same_taskTMaze_differentDay.npy", assignments)

# Save matchings using pickle to handle the list of lists structure
with open(f"{save_dir}matching_Round_same_taskTMaze_differentDay.pkl", "wb") as f:
	pickle.dump(matchings, f)

In [ ]:
# Filter components by number of sessions the component could be found
n_reg = len(file_paths) # minimal number of sessions that each component has to be registered in

# Use number of non-NaNs in each row to filter out components that were not registered in enough sessions
assignments_filtered = np.array(np.nan_to_num(assignments[np.sum(~np.isnan(assignments), axis=1) >= n_reg]), dtype=int);
print(len(assignments_filtered))
# Ensure indices are within the valid range
valid_indices = assignments_filtered[:, 0] < spatial[-1].shape[1]
assignments_filtered = assignments_filtered[valid_indices]

# choose template for visualization
template_no = -1

# Use filtered indices to select the corresponding spatial components
spatial_filtered = spatial[template_no][:, assignments_filtered[:, template_no]]

# Plot spatial components of the selected components on the template of the last session
visualization.plot_contours(spatial_filtered, templates[template_no]);

In [ ]:
# load data
cnmf_list, spatial, templates, dims, idx_components = load_cnmf_data(same_taskLinear_differentDay)
# multisession registration
spatial_union, assignments, matchings = register_multisession(A=spatial, dims=dims, templates=templates)
# Save results
save_dir = f"/media/toor/T7Shield/Mouse1637/Registration_CrossTask_Tests/"

# Ensure the directory exists
os.makedirs(save_dir, exist_ok=True)

np.save(f"{save_dir}spatial_union_same_taskLinear_differentDay.npy", spatial_union)
np.save(f"{save_dir}assignments_same_taskLinear_differentDay.npy", assignments)

# Save matchings using pickle to handle the list of lists structure
with open(f"{save_dir}matching_Round_same_taskLinear_differentDay.pkl", "wb") as f:
	pickle.dump(matchings, f)

In [ ]:
# Filter components by number of sessions the component could be found
n_reg = len(file_paths) # minimal number of sessions that each component has to be registered in

# Use number of non-NaNs in each row to filter out components that were not registered in enough sessions
assignments_filtered = np.array(np.nan_to_num(assignments[np.sum(~np.isnan(assignments), axis=1) >= n_reg]), dtype=int);
print(len(assignments_filtered))
# Ensure indices are within the valid range
valid_indices = assignments_filtered[:, 0] < spatial[-1].shape[1]
assignments_filtered = assignments_filtered[valid_indices]

# choose template for visualization
template_no = -1

# Use filtered indices to select the corresponding spatial components
spatial_filtered = spatial[template_no][:, assignments_filtered[:, template_no]]

# Plot spatial components of the selected components on the template of the last session
visualization.plot_contours(spatial_filtered, templates[template_no]);

In [ ]:
# load data
cnmf_list, spatial, templates, dims, idx_components = load_cnmf_data(cross_task_sameDay)
# multisession registration
spatial_union, assignments, matchings = register_multisession(A=spatial, dims=dims, templates=templates)
# Save results
save_dir = f"/media/toor/T7Shield/Mouse1637/Registration_CrossTask_Tests/"

# Ensure the directory exists
os.makedirs(save_dir, exist_ok=True)

np.save(f"{save_dir}spatial_union_cross_task_sameDay.npy", spatial_union)
np.save(f"{save_dir}assignments_cross_task_sameDay.npy", assignments)

# Save matchings using pickle to handle the list of lists structure
with open(f"{save_dir}matching_Round_cross_task_sameDay.pkl", "wb") as f:
	pickle.dump(matchings, f)

In [ ]:
# Filter components by number of sessions the component could be found
n_reg = len(file_paths) # minimal number of sessions that each component has to be registered in

# Use number of non-NaNs in each row to filter out components that were not registered in enough sessions
assignments_filtered = np.array(np.nan_to_num(assignments[np.sum(~np.isnan(assignments), axis=1) >= n_reg]), dtype=int);
print(len(assignments_filtered))
# Ensure indices are within the valid range
valid_indices = assignments_filtered[:, 0] < spatial[-1].shape[1]
assignments_filtered = assignments_filtered[valid_indices]

# choose template for visualization
template_no = -1

# Use filtered indices to select the corresponding spatial components
spatial_filtered = spatial[template_no][:, assignments_filtered[:, template_no]]

# Plot spatial components of the selected components on the template of the last session
visualization.plot_contours(spatial_filtered, templates[template_no]);

In [ ]:
# load data
cnmf_list, spatial, templates, dims, idx_components = load_cnmf_data(cross_task_differentDay)
# multisession registration
spatial_union, assignments, matchings = register_multisession(A=spatial, dims=dims, templates=templates)
# Save results
save_dir = f"/media/toor/T7Shield/Mouse1637/Registration_CrossTask_Tests/"

# Ensure the directory exists
os.makedirs(save_dir, exist_ok=True)

np.save(f"{save_dir}spatial_union_cross_task_differentDay.npy", spatial_union)
np.save(f"{save_dir}assignments_cross_task_differentDay.npy", assignments)

# Save matchings using pickle to handle the list of lists structure
with open(f"{save_dir}matching_Round_cross_task_differentDay.pkl", "wb") as f:
	pickle.dump(matchings, f)

In [ ]:
# Filter components by number of sessions the component could be found
n_reg = len(file_paths) # minimal number of sessions that each component has to be registered in

# Use number of non-NaNs in each row to filter out components that were not registered in enough sessions
assignments_filtered = np.array(np.nan_to_num(assignments[np.sum(~np.isnan(assignments), axis=1) >= n_reg]), dtype=int);
print(len(assignments_filtered))
# Ensure indices are within the valid range
valid_indices = assignments_filtered[:, 0] < spatial[-1].shape[1]
assignments_filtered = assignments_filtered[valid_indices]

# choose template for visualization
template_no = -1

# Use filtered indices to select the corresponding spatial components
spatial_filtered = spatial[template_no][:, assignments_filtered[:, template_no]]

# Plot spatial components of the selected components on the template of the last session
visualization.plot_contours(spatial_filtered, templates[template_no]);